In [1]:
import pandas as pd
print("Pandas version:", pd.__version__)


Pandas version: 2.3.3


In [2]:
import pandas as pd

df_customers = pd.read_csv("customers_raw.csv")
df_orders = pd.read_csv("orders_raw.csv")

print("Customers shape:", df_customers.shape)
print("Orders shape:", df_orders.shape)

df_customers.head()


Customers shape: (525, 6)
Orders shape: (1500, 5)


,customer_id,name,email,phone,country,signup_date
0,CUST00001,Allison Hill,donaldgarcia@example.net,+1-219-560-0133,United States,2025-11-01
1,CUST00002,Jennifer Cole,lisa02@example.net,(254)923-5116x15594,USA,2025-04-18
2,CUST00003,Jamie Arnold,barbara10@example.net,441.731.6475,United Kingdom,2024-10-02
3,CUST00004,Mia Sutton,lynchgeorge@example.net,527.264.8350,Canada,2026-02-07
4,CUST00005,Austin Gentry,NaN,NaN,Canada,2026-09-06


In [3]:
# ---- COMPLETENESS CHECK ----
print("\n--- Missing Values: Customers ---")
missing_customers = df_customers.isnull().sum()
missing_pct = (missing_customers / len(df_customers) * 100).round(2)
completeness_report = pd.DataFrame({
    "missing_count": missing_customers,
    "missing_pct": missing_pct
})
print(completeness_report)

print("\n--- Missing Values: Orders ---")
missing_orders = df_orders.isnull().sum()
print(missing_orders)



--- Missing Values: Customers ---
             missing_count  missing_pct
customer_id              0         0.00
name                     0         0.00
email                   78        14.86
phone                   78        14.86
country                  0         0.00
signup_date              0         0.00

--- Missing Values: Orders ---
order_id       0
customer_id    0
order_date     0
quantity       0
unit_price     0
dtype: int64


In [4]:
# ---- COMPLETENESS SCORE ----
total_cells = df_customers.shape[0] * df_customers.shape[1]
total_missing = df_customers.isnull().sum().sum()

completeness_score = round((1 - total_missing / total_cells) * 100, 2)
print(f"Overall Customer Table Completeness Score: {completeness_score}%")

# Per-column completeness (more useful for reporting — shows WHERE the problem is)
column_completeness = (100 - missing_pct).rename("completeness_%")
print("\nPer-column completeness:")
print(column_completeness)


Overall Customer Table Completeness Score: 95.05%

Per-column completeness:
customer_id    100.00
name           100.00
email           85.14
phone           85.14
country        100.00
signup_date    100.00
Name: completeness_%, dtype: float64


In [5]:
# ---- UNIQUENESS CHECK: Pass 1 - Normalization ----
df_customers["name_normalized"] = df_customers["name"].str.strip().str.lower()

duplicate_mask = df_customers.duplicated(subset=["name_normalized"], keep=False)
duplicates_found = df_customers[duplicate_mask].sort_values("name_normalized")

print(f"Duplicate records found after normalization: {duplicate_mask.sum()}")
duplicates_found[["customer_id", "name", "name_normalized", "email"]].head(10)


Duplicate records found after normalization: 53


,customer_id,name,name_normalized,email
509,CUST90009,Amber Gomez,amber gomez,nhuber@example.org
314,CUST00315,Amber Gomez,amber gomez,nhuber@example.org
385,CUST00386,Andrea Davis,andrea davis,NaN
71,CUST00072,Andrea Davis,andrea davis,umitchell@example.net
501,CUST90001,Andrea Rodriguez,andrea rodriguez,pmurphy@example.net
280,CUST00281,Andrea Rodriguez,andrea rodriguez,pmurphy@example.net
502,CUST90002,Andrew Allen,andrew allen,mturner@example.org
440,CUST00441,Andrew Allen,andrew allen,mturner@example.org
504,CUST90004,BETHANY FOX,bethany fox,imendoza@example.org
225,CUST00226,Bethany Fox,bethany fox,imendoza@example.org


In [6]:
# ---- UNIQUENESS CHECK: Confidence scoring ----
dup_groups = df_customers[duplicate_mask].groupby("name_normalized")

high_confidence = []
needs_review = []

for name, group in dup_groups:
    emails = group["email"].dropna().unique()
    if len(emails) <= 1:
        # same email (or all missing) -> high confidence same person
        high_confidence.append(name)
    else:
        # different emails -> could be two different people, flag for review
        needs_review.append(name)

print(f"High-confidence duplicate groups (matching email): {len(high_confidence)}")
print(f"Needs manual review (name matches, email differs/uncertain): {len(needs_review)}")
print("\nGroups needing review:", needs_review)

High-confidence duplicate groups (matching email): 25
Needs manual review (name matches, email differs/uncertain): 1

Groups needing review: ['laura smith']


In [7]:
from rapidfuzz import fuzz

# Only check names NOT already caught by exact normalization, to avoid redundant work
already_flagged = set(df_customers[duplicate_mask]["name_normalized"])
remaining = df_customers[~df_customers["name_normalized"].isin(already_flagged)].copy()
names = remaining["name_normalized"].tolist()
ids = remaining["customer_id"].tolist()

fuzzy_pairs = []
SIMILARITY_THRESHOLD = 90  # out of 100 - tune this based on false positive rate

for i in range(len(names)):
    for j in range(i + 1, len(names)):
        score = fuzz.ratio(names[i], names[j])
        if score >= SIMILARITY_THRESHOLD:
            fuzzy_pairs.append((ids[i], names[i], ids[j], names[j], score))

print(f"Fuzzy-matched near-duplicate pairs found: {len(fuzzy_pairs)}")
for pair in fuzzy_pairs[:10]:
    print(pair)

Fuzzy-matched near-duplicate pairs found: 1
('CUST00010', 'austin johnson', 'CUST00316', 'justin johnson', 92.85714285714286)


In [8]:
# ---- UNIQUENESS SCORE ----
exact_duplicates = duplicate_mask.sum()  # 53 rows involved in exact/normalized dupes
fuzzy_duplicates = len(fuzzy_pairs)      # 1 pair (2 rows), pending manual review

total_customers = len(df_customers)
uniqueness_score = round((1 - (exact_duplicates / total_customers)) * 100, 2)

print(f"Uniqueness Score: {uniqueness_score}%")
print(f"Exact/normalized duplicate records: {exact_duplicates}")
print(f"Fuzzy-matched candidates (needs review): {fuzzy_duplicates}")


Uniqueness Score: 89.9%
Exact/normalized duplicate records: 53
Fuzzy-matched candidates (needs review): 1


In [9]:
# ---- VALIDITY CHECK: Date formats ----
import re

def detect_date_format(date_str):
    if re.match(r'^\d{4}-\d{2}-\d{2}$', date_str):
        return 'YYYY-MM-DD'
    elif re.match(r'^\d{2}/\d{2}/\d{4}$', date_str):
        return 'DD/MM/YYYY'
    elif re.match(r'^\d{2}-\d{2}-\d{4}$', date_str):
        return 'MM-DD-YYYY'
    else:
        return 'UNKNOWN'

df_orders["date_format_detected"] = df_orders["order_date"].apply(detect_date_format)

format_counts = df_orders["date_format_detected"].value_counts()
format_pct = (format_counts / len(df_orders) * 100).round(2)

print("Date format distribution:")
print(pd.DataFrame({"count": format_counts, "pct": format_pct}))

Date format distribution:
                      count    pct
date_format_detected              
YYYY-MM-DD              523  34.87
MM-DD-YYYY              501  33.40
DD/MM/YYYY              476  31.73


In [10]:
# ---- VALIDITY FIX: Parse each row with its correct format ----
format_map = {
    'YYYY-MM-DD': '%Y-%m-%d',
    'MM-DD-YYYY': '%m-%d-%Y',
    'DD/MM/YYYY': '%d/%m/%Y'
}

def parse_with_detected_format(row):
    fmt = format_map.get(row["date_format_detected"])
    if fmt is None:
        return pd.NaT  # couldn't detect format -> treat as invalid, don't guess
    try:
        return pd.to_datetime(row["order_date"], format=fmt)
    except ValueError:
        return pd.NaT  # value present but doesn't actually match the format (e.g. day=35)

df_orders["order_date_parsed"] = df_orders.apply(parse_with_detected_format, axis=1)

invalid_dates = df_orders["order_date_parsed"].isna().sum()
print(f"Dates that failed to parse even with detection: {invalid_dates}")
df_orders[["order_id", "order_date", "date_format_detected", "order_date_parsed"]].head(10)

Dates that failed to parse even with detection: 0


,order_id,order_date,date_format_detected,order_date_parsed
0,ORD000001,09-29-2024,MM-DD-YYYY,2024-09-29
1,ORD000002,18/12/2024,DD/MM/YYYY,2024-12-18
2,ORD000003,07/11/2024,DD/MM/YYYY,2024-11-07
3,ORD000004,2026-01-26,YYYY-MM-DD,2026-01-26
4,ORD000005,09/12/2024,DD/MM/YYYY,2024-12-09
5,ORD000006,03-27-2025,MM-DD-YYYY,2025-03-27
6,ORD000007,2025-01-31,YYYY-MM-DD,2025-01-31
7,ORD000008,14/11/2024,DD/MM/YYYY,2024-11-14
8,ORD000009,07/02/2026,DD/MM/YYYY,2026-02-07
9,ORD000010,2025-12-28,YYYY-MM-DD,2025-12-28


In [11]:
# ---- VALIDITY CHECK: Quantity and price sanity ----
invalid_quantity = df_orders[df_orders["quantity"] <= 0]
print(f"Orders with invalid (zero/negative) quantity: {len(invalid_quantity)}")
print(invalid_quantity[["order_id", "customer_id", "quantity"]].head())

# Outlier detection using IQR method (standard statistical approach, not just eyeballing)
Q1 = df_orders["unit_price"].quantile(0.25)
Q3 = df_orders["unit_price"].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR

price_outliers = df_orders[df_orders["unit_price"] > upper_bound]
print(f"\nPrice outliers (unit_price > {upper_bound:.2f}): {len(price_outliers)}")
print(price_outliers[["order_id", "unit_price"]].sort_values("unit_price", ascending=False).head())

Orders with invalid (zero/negative) quantity: 33
      order_id customer_id  quantity
57   ORD000058   CUST00375        -7
103  ORD000104   CUST00096        -6
114  ORD000115   CUST00481        -2
170  ORD000171   CUST00484        -3
178  ORD000179   CUST00190        -4

Price outliers (unit_price > 751.13): 12
       order_id  unit_price
544   ORD000545    445710.0
30    ORD000031    413340.0
744   ORD000745    378240.0
697   ORD000698    353430.0
1006  ORD001007    331810.0


In [12]:
# ---- VALIDITY SCORE ----
total_orders = len(df_orders)
invalid_dates_count = df_orders["order_date_parsed"].isna().sum()
invalid_qty_count = len(invalid_quantity)
outlier_count = len(price_outliers)

total_invalid_flags = invalid_dates_count + invalid_qty_count  # outliers are a separate concern, not "invalid" per se
validity_score = round((1 - total_invalid_flags / total_orders) * 100, 2)

print(f"Validity Score: {validity_score}%")
print(f"  - Invalid dates: {invalid_dates_count}")
print(f"  - Invalid quantities: {invalid_qty_count}")
print(f"  - Price outliers (flagged separately, not counted against score): {outlier_count}")

Validity Score: 97.8%
  - Invalid dates: 0
  - Invalid quantities: 33
  - Price outliers (flagged separately, not counted against score): 12


In [13]:
# ---- CONSISTENCY CHECK: Country naming ----
print("Raw country value counts:")
print(df_customers["country"].value_counts())

Raw country value counts:
country
Canada            81
Germany           74
United States     70
United Kingdom    66
USA               63
US                59
Australia         59
UK                53
Name: count, dtype: int64


In [14]:
# ---- CONSISTENCY FIX: Standardization mapping ----
country_mapping = {
    "USA": "United States",
    "US": "United States",
    "United States": "United States",
    "UK": "United Kingdom",
    "United Kingdom": "United Kingdom",
    "Canada": "Canada",
    "Germany": "Germany",
    "Australia": "Australia"
}

df_customers["country_standardized"] = df_customers["country"].map(country_mapping)

# Sanity check: did every value map successfully?
unmapped = df_customers[df_customers["country_standardized"].isna()]
print(f"Unmapped country values (should be 0): {len(unmapped)}")

print("\nStandardized country value counts:")
print(df_customers["country_standardized"].value_counts())

Unmapped country values (should be 0): 0

Standardized country value counts:
country_standardized
United States     192
United Kingdom    119
Canada             81
Germany            74
Australia          59
Name: count, dtype: int64


In [15]:
# ---- CONSISTENCY SCORE ----
# Consistency = % of records that were already in their standardized/canonical form
already_consistent = (df_customers["country"] == df_customers["country_standardized"]).sum()
total_records = len(df_customers)

consistency_score = round((already_consistent / total_records) * 100, 2)
print(f"Consistency Score: {consistency_score}%")
print(f"Records requiring standardization: {total_records - already_consistent}")

Consistency Score: 66.67%
Records requiring standardization: 175


In [16]:
# ---- ACCURACY (PROXY): Email format validity ----
import re

email_pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'

def is_plausible_email(email):
    if pd.isna(email):
        return None  # missing is a Completeness issue, not an Accuracy one - don't double-count
    return bool(re.match(email_pattern, email))

df_customers["email_plausible"] = df_customers["email"].apply(is_plausible_email)

valid_emails = df_customers["email_plausible"].sum()
checked_emails = df_customers["email_plausible"].notna().sum()

print(f"Plausibly valid emails: {valid_emails} / {checked_emails} checked")
print(df_customers[df_customers["email_plausible"] == False][["customer_id", "email"]])

Plausibly valid emails: 447 / 447 checked
Empty DataFrame
Columns: [customer_id, email]
Index: []


In [17]:
# ---- ACCURACY (PROXY): Phone format validity ----
def is_plausible_phone(phone):
    if pd.isna(phone):
        return None
    digit_count = sum(c.isdigit() for c in phone)
    return 7 <= digit_count <= 15  # international phone numbers typically fall in this digit range

df_customers["phone_plausible"] = df_customers["phone"].apply(is_plausible_phone)

valid_phones = df_customers["phone_plausible"].sum()
checked_phones = df_customers["phone_plausible"].notna().sum()

print(f"Plausibly valid phones: {valid_phones} / {checked_phones} checked")
print(df_customers[df_customers["phone_plausible"] == False][["customer_id", "phone"]].head())

Plausibly valid phones: 377 / 447 checked
   customer_id                  phone
13   CUST00014  +1-736-619-3990x91699
23   CUST00024   001-967-263-2016x328
54   CUST00055   001-786-592-6179x640
57   CUST00058  001-617-975-8917x8390
64   CUST00065  +1-497-684-0369x00343


In [18]:
# ---- ACCURACY (PROXY): Phone check, refined to handle extensions ----
def is_plausible_phone_v2(phone):
    if pd.isna(phone):
        return None
    main_number = re.split(r'[xX]', phone)[0]  # drop extension before counting
    digit_count = sum(c.isdigit() for c in main_number)
    return 7 <= digit_count <= 15

df_customers["phone_plausible"] = df_customers["phone"].apply(is_plausible_phone_v2)

valid_phones = df_customers["phone_plausible"].sum()
checked_phones = df_customers["phone_plausible"].notna().sum()

print(f"Plausibly valid phones (refined): {valid_phones} / {checked_phones} checked")
print(df_customers[df_customers["phone_plausible"] == False][["customer_id", "phone"]].head())

Plausibly valid phones (refined): 447 / 447 checked
Empty DataFrame
Columns: [customer_id, phone]
Index: []


In [19]:
# ---- ACCURACY SCORE ----
total_checkable = checked_emails + checked_phones
total_valid = valid_emails + valid_phones

accuracy_score = round((total_valid / total_checkable) * 100, 2)
print(f"Accuracy Score (proxy-based): {accuracy_score}%")


Accuracy Score (proxy-based): 100.0%


In [20]:
# ---- TIMELINESS CHECK ----
df_customers["signup_date"] = pd.to_datetime(df_customers["signup_date"])
today = pd.Timestamp.now().normalize()

future_signups = df_customers[df_customers["signup_date"] > today]
print(f"Customers with future signup dates: {len(future_signups)}")
print(future_signups[["customer_id", "signup_date"]].head(10))

# Also flag very old records that might be stale (e.g. >3 years, arbitrary business rule)
oldest_allowed = today - pd.Timedelta(days=3*365)
stale_records = df_customers[df_customers["signup_date"] < oldest_allowed]
print(f"\nRecords older than 3 years (potentially stale): {len(stale_records)}")

Customers with future signup dates: 0
Empty DataFrame
Columns: [customer_id, signup_date]
Index: []

Records older than 3 years (potentially stale): 0


In [21]:
# ---- TIMELINESS SCORE ----
total_customers = len(df_customers)
timeliness_issues = len(future_signups) + len(stale_records)
timeliness_score = round((1 - timeliness_issues / total_customers) * 100, 2)
print(f"Timeliness Score: {timeliness_score}%")

Timeliness Score: 100.0%


In [22]:
# ---- OVERALL DATA QUALITY SCORE ----
dq_scores = {
    "Completeness": completeness_score,
    "Uniqueness": uniqueness_score,
    "Validity": validity_score,
    "Consistency": consistency_score,
    "Accuracy": accuracy_score,
    "Timeliness": timeliness_score
}

dq_scorecard = pd.DataFrame(list(dq_scores.items()), columns=["Dimension", "Score"])
overall_dq_score = round(dq_scorecard["Score"].mean(), 2)

print(dq_scorecard)
print(f"\nOVERALL DATA QUALITY SCORE: {overall_dq_score}%")

      Dimension   Score
0  Completeness   95.05
1    Uniqueness   89.90
2      Validity   97.80
3   Consistency   66.67
4      Accuracy  100.00
5    Timeliness  100.00

OVERALL DATA QUALITY SCORE: 91.57%


In [23]:
# ---- BUSINESS IMPACT: Duplicate customer cost ----
duplicate_customer_count = 53  # from our uniqueness check
avg_marketing_cost_per_customer_per_campaign = 2.50  # assumption: $2.50/customer for a typical email+retargeting campaign
campaigns_per_year = 12

wasted_marketing_spend = duplicate_customer_count * avg_marketing_cost_per_customer_per_campaign * campaigns_per_year
print(f"Estimated annual wasted marketing spend from duplicates: ${wasted_marketing_spend:,.2f}")

Estimated annual wasted marketing spend from duplicates: $1,590.00


In [24]:
# ---- BUSINESS IMPACT: Missing contact info ----
missing_contact_count = 78  # customers missing email or phone (from completeness check)
avg_customer_annual_value = 150  # assumption: average revenue per customer per year
reactivation_campaign_reach_loss_pct = 0.30  # assumption: 30% less likely to convert without direct contact channel

lost_revenue_opportunity = missing_contact_count * avg_customer_annual_value * reactivation_campaign_reach_loss_pct
print(f"Estimated at-risk revenue from unreachable customers: ${lost_revenue_opportunity:,.2f}")

Estimated at-risk revenue from unreachable customers: $3,510.00


In [25]:
# ---- BUSINESS IMPACT: Consistency issue → reporting error ----
# Before fix: US customers appeared as 3 separate segments (70+63+59=192, but reported as fragments)
us_fragmented_reporting = 70  # what a naive report would show as "United States"
us_true_count = 192  # what we now know is correct

understatement_pct = round((1 - us_fragmented_reporting / us_true_count) * 100, 1)
print(f"US customer base was understated by {understatement_pct}% in any report using raw country field")
print("Business risk: regional budget/staffing/inventory decisions based on this number would be wrong")

US customer base was understated by 63.5% in any report using raw country field
Business risk: regional budget/staffing/inventory decisions based on this number would be wrong
